In [ ]:
import numpy as np
from itertools import product

def generate_matrix(n_agents=3, n_tasks=6, total=50):
    matrix = []
    for _ in range(n_agents):
        cuts = sorted(np.random.choice(range(1, total), n_tasks - 1, replace=False))
        cuts = [0] + cuts + [total]
        row = [cuts[i+1] - cuts[i] for i in range(n_tasks)]
        matrix.append(row)
    return np.array(matrix)

def get_all_allocations(n_agents=3, n_tasks=6):
    all_allocs = []
    for assignment in product(range(n_agents), repeat=n_tasks):
        if len(set(assignment)) == n_agents:
            all_allocs.append(assignment)
    return all_allocs

def agent_costs(matrix, allocation):
    n_agents = matrix.shape[0]
    costs = [0] * n_agents
    for task, agent in enumerate(allocation):
        costs[agent] += matrix[agent][task]
    return costs

def is_pareto_optimal(matrix, allocation, all_allocs):
    costs = agent_costs(matrix, allocation)
    for other in all_allocs:
        other_costs = agent_costs(matrix, other)
        if all(other_costs[i] <= costs[i] for i in range(len(costs))) and \
           any(other_costs[i] < costs[i] for i in range(len(costs))):
            return False
    return True

def usw_allocation(matrix, all_allocs):
    best_usw = float('inf')
    best = []
    for alloc in all_allocs:
        costs = agent_costs(matrix, alloc)
        total_cost = sum(costs)
        if total_cost < best_usw:
            best_usw = total_cost
            best = [alloc]
        elif total_cost == best_usw:
            best.append(alloc)
    return best, best_usw

def rmm_allocation(matrix, all_allocs):
    best_max = float('inf')
    best = []
    for alloc in all_allocs:
        costs = agent_costs(matrix, alloc)
        max_cost = max(costs)
        if max_cost < best_max:
            best_max = max_cost
            best = [alloc]
        elif max_cost == best_max:
            best.append(alloc)
    return best, best_max

def is_envy_free(matrix, allocation):
    n_agents = matrix.shape[0]
    for i in range(n_agents):
        my_cost = sum(matrix[i][t] for t, a in enumerate(allocation) if a == i)
        for j in range(n_agents):
            if i == j:
                continue
            j_bundle_cost_for_i = sum(matrix[i][t] for t, a in enumerate(allocation) if a == j)
            if j_bundle_cost_for_i < my_cost:
                return False
    return True

def is_exact_eq(matrix, allocation):
    costs = agent_costs(matrix, allocation)
    return len(set(costs)) == 1

def is_approx_eq(matrix, allocation):
    costs = agent_costs(matrix, allocation)
    return max(costs) - min(costs) <= 1

def is_approx_not_exact_eq(matrix, allocation):
    """spread = exactly 1: approx-EQ but NOT exact-EQ."""
    costs = agent_costs(matrix, allocation)
    return max(costs) - min(costs) == 1

def find_valid_instance(max_attempts=100000):
    """
    Find an instance where, across ALL allocations:
      - USW, RMM, and approx-not-exact-EQ+RMM+EF sets are pairwise disjoint
      - All USW-optimal allocations are PO, NOT EF, NOT approx-EQ
      - All RMM-optimal allocations are PO, NOT USW, NOT EF, NOT approx-EQ
      - All approx-not-exact-EQ+RMM+EF allocations are PO, NOT USW, NOT exact-EQ

    And there exists at least one candidate in each of the three roles:
      1. USW + PO                                  — NOT RMM, NOT EF, NOT approx-EQ
      2. RMM + PO                                  — NOT USW, NOT EF, NOT approx-EQ
      3. approx-EQ (spread=1) + RMM + EF + PO      — NOT USW, NOT exact-EQ
    All three chosen allocations must be mutually distinct.
    """
    all_allocs = get_all_allocations()
    print(f"Total allocations to consider: {len(all_allocs)}")

    for attempt in range(max_attempts):
        if (attempt + 1) % 5000 == 0:
            print(f"Attempt {attempt + 1}...")

        matrix = generate_matrix()

        usw_allocs, best_usw = usw_allocation(matrix, all_allocs)
        rmm_allocs, best_max = rmm_allocation(matrix, all_allocs)
        ef_allocs  = [a for a in all_allocs if is_envy_free(matrix, a)]
        po_allocs  = [a for a in all_allocs if is_pareto_optimal(matrix, a, all_allocs)]

        # approx-not-exact-EQ + RMM + EF + PO candidates
        aneq_rmm_ef_po_allocs = [
            a for a in all_allocs
            if is_approx_not_exact_eq(matrix, a)
            and a in rmm_allocs
            and is_envy_free(matrix, a)
            and a in po_allocs
        ]

        if not ef_allocs:
            continue
        if not aneq_rmm_ef_po_allocs:
            continue

        # --- All USW allocations must be PO, NOT EF, NOT approx-EQ ---
        if any(a not in po_allocs for a in usw_allocs):
            continue
        if any(is_envy_free(matrix, a) for a in usw_allocs):
            continue
        if any(is_approx_eq(matrix, a) for a in usw_allocs):
            continue

        # --- All RMM allocations must be PO, NOT USW, NOT EF, NOT approx-EQ ---
        # (RMM ∩ USW = ∅ checked below; here we check PO, EF, approx-EQ)
        pure_rmm_allocs = [a for a in rmm_allocs if a not in aneq_rmm_ef_po_allocs]
        if not pure_rmm_allocs:
            continue
        if any(a not in po_allocs for a in pure_rmm_allocs):
            continue
        if any(is_envy_free(matrix, a) for a in pure_rmm_allocs):
            continue
        if any(is_approx_eq(matrix, a) for a in pure_rmm_allocs):
            continue

        # --- Global set-level disjointness ---
        if any(a in rmm_allocs for a in usw_allocs):           # USW ∩ RMM = ∅
            continue
        if any(a in aneq_rmm_ef_po_allocs for a in usw_allocs): # USW ∩ aneq_set = ∅
            continue

        # --- Pick one representative from each set ---
        usw_alloc      = usw_allocs[0]
        rmm_alloc      = rmm_allocs[0]
        aneq_alloc     = aneq_rmm_ef_po_allocs[0]

        if len({usw_alloc, rmm_alloc, aneq_alloc}) < 3:
            continue

        # --- Success ---
        print(f"\nValid instance found after {attempt + 1} attempts!\n")
        print("Disutility Matrix (agents x tasks):")
        print(matrix)
        print()

        usw_costs  = agent_costs(matrix, usw_alloc)
        rmm_costs  = agent_costs(matrix, rmm_alloc)
        aneq_costs = agent_costs(matrix, aneq_alloc)

        print(f"1. USW + PO  (total cost = {sum(usw_costs)})  |  NOT RMM / NOT EF / NOT approx-EQ")
        print(f"   Task assignment: {usw_alloc}")
        print(f"   Agent costs:     {usw_costs}")
        print()

        print(f"2. RMM + PO  (max cost = {best_max})  |  NOT USW / NOT EF / NOT approx-EQ")
        print(f"   Task assignment: {rmm_alloc}")
        print(f"   Agent costs:     {rmm_costs}")
        print()

        print(f"3. approx-EQ (spread=1) + RMM + EF + PO  |  NOT USW / NOT exact-EQ")
        print(f"   Task assignment: {aneq_alloc}")
        print(f"   Agent costs:     {aneq_costs}  (spread = {max(aneq_costs) - min(aneq_costs)})")
        print()

        print(f"All three allocations are distinct: {len({usw_alloc, rmm_alloc, aneq_alloc}) == 3}")
        return matrix, usw_alloc, rmm_alloc, aneq_alloc

    print("No valid instance found within attempt limit.")
    return None


instances3 = []
for _ in range(10):
    result = find_valid_instance()
    if result is not None:
        instances3.append(result)